# T35 - 不同曲线投资顺序分析
## 第7章：报告汇总

### 概述
本章汇总所有分析结果，生成最终报告，包括：
1. 加载所有分析结果
2. 生成各维度分析报告
3. 汇总最优策略
4. 生成JSON格式报告

In [ ]:
# 1. 导入必要的库
import pandas as pd
import numpy as np
from datetime import datetime
import os
import json
import warnings

warnings.filterwarnings('ignore')

# 设置显示选项
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("库导入成功！")

In [ ]:
# 2. 加载配置
from config import (
    OUTPUT_DIR,
    YIELD_DATA_FILE,
    CYCLE_LABEL_FILE,
    YIELD_DOWN_CYCLES,
    INITIAL_AMOUNT
)

print(f"输出目录: {OUTPUT_DIR}")

In [ ]:
# 3. 加载所有分析结果

def load_all_results():
    """加载所有分析结果"""
    print("加载分析结果...")
    print("=" * 60)
    
    results = {}
    dimensions = ['品种维度', '久期维度', '资质维度']
    
    for dimension in dimensions:
        # 加载回测结果
        backtest_file = os.path.join(OUTPUT_DIR, f'{dimension}回测结果.csv')
        if os.path.exists(backtest_file):
            df = pd.read_csv(backtest_file, encoding='utf-8-sig')
            results[f'{dimension}_backtest'] = df
            print(f"  加载 {dimension}回测结果: {len(df)} 条记录")
        
        # 加载策略统计
        stats_file = os.path.join(OUTPUT_DIR, f'{dimension}策略统计.csv')
        if os.path.exists(stats_file):
            df = pd.read_csv(stats_file, encoding='utf-8-sig', index_col=0)
            results[f'{dimension}_stats'] = df
            print(f"  加载 {dimension}策略统计: {len(df)} 个策略")
    
    # 加载周期标注
    if os.path.exists(CYCLE_LABEL_FILE):
        cycles_df = pd.read_csv(CYCLE_LABEL_FILE, encoding='utf-8-sig')
        results['cycles'] = cycles_df
        print(f"  加载周期标注: {len(cycles_df)} 个周期")
    
    return results

all_results = load_all_results()

In [ ]:
# 4. 生成各维度分析报告

def generate_dimension_reports(all_results):
    """生成各维度分析报告"""
    print("生成各维度分析报告...")
    print("=" * 60)
    
    dimensions = ['品种维度', '久期维度', '资质维度']
    
    for dimension in dimensions:
        backtest_key = f'{dimension}_backtest'
        stats_key = f'{dimension}_stats'
        
        if backtest_key not in all_results or stats_key not in all_results:
            continue
        
        backtest_df = all_results[backtest_key]
        stats_df = all_results[stats_key]
        
        # 生成报告文件
        report_file = os.path.join(OUTPUT_DIR, f'{dimension}策略分析报告.txt')
        
        with open(report_file, 'w', encoding='utf-8') as f:
            f.write(f"{dimension}策略分析报告\n")
            f.write("=" * 50 + "\n\n")
            f.write(f"生成时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
            
            # 1. 策略统计汇总
            f.write("1. 策略统计汇总\n")
            f.write("-" * 30 + "\n\n")
            
            # 按平均年化收益率排序
            sorted_stats = stats_df.sort_values('年化收益率均值', ascending=False)
            
            for i, (strategy, row) in enumerate(sorted_stats.iterrows(), 1):
                f.write(f"{i}. {strategy}\n")
                f.write(f"   平均年化收益率: {row['年化收益率均值']:.2%}\n")
                f.write(f"   收益率波动率: {row['年化收益率标准差']:.2%}\n")
                f.write(f"   最大年化收益率: {row['年化收益率最大值']:.2%}\n")
                f.write(f"   最小年化收益率: {row['年化收益率最小值']:.2%}\n")
                f.write(f"\n")
            
            # 2. 各周期最优策略
            f.write("\n2. 各周期最优策略\n")
            f.write("-" * 30 + "\n\n")
            
            cycle_performance = backtest_df.pivot_table(
                index='cycle_id',
                columns='strategy_sequence',
                values='annual_return',
                aggfunc='mean'
            )
            
            for cycle_id in cycle_performance.index:
                period_returns = cycle_performance.loc[cycle_id]
                best_strategy = period_returns.idxmax()
                best_return = period_returns.max()
                
                # 获取周期信息
                if 'cycles' in all_results:
                    cycle_info = all_results['cycles'][all_results['cycles']['cycle'] == cycle_id].iloc[0]
                    f.write(f"周期 {cycle_id} ({cycle_info['start_date']} 至 {cycle_info['end_date']})\n")
                    f.write(f"  收益率变动: {cycle_info['start_yield']:.2f}% -> {cycle_info['end_yield']:.2f}%\n")
                else:
                    f.write(f"周期 {cycle_id}\n")
                
                f.write(f"  最优策略: {best_strategy}\n")
                f.write(f"  年化收益率: {best_return:.2%}\n\n")
            
            # 3. 策略稳定性排名
            f.write("\n3. 策略稳定性排名（按波动率从低到高）\n")
            f.write("-" * 30 + "\n\n")
            
            stability_ranking = stats_df['年化收益率标准差'].sort_values()
            
            for rank, (strategy, std) in enumerate(stability_ranking.items(), 1):
                mean_return = stats_df.loc[strategy, '年化收益率均值']
                f.write(f"{rank}. {strategy}\n")
                f.write(f"   平均收益率: {mean_return:.2%}\n")
                f.write(f"   波动率: {std:.2%}\n\n")
        
        print(f"  生成 {dimension}策略分析报告.txt")

generate_dimension_reports(all_results)

In [ ]:
# 5. 找出每个维度的最优策略

def find_best_strategies(all_results):
    """找出每个维度的最优策略"""
    print("找出最优策略...")
    print("=" * 60)
    
    best_strategies = {}
    dimensions = ['品种维度', '久期维度', '资质维度']
    
    for dimension in dimensions:
        stats_key = f'{dimension}_stats'
        
        if stats_key not in all_results:
            continue
        
        stats_df = all_results[stats_key]
        
        # 按平均年化收益率排序
        best_strategy = stats_df['年化收益率均值'].idxmax()
        best_return = stats_df.loc[best_strategy, '年化收益率均值']
        best_std = stats_df.loc[best_strategy, '年化收益率标准差']
        
        best_strategies[dimension] = {
            'strategy': best_strategy,
            'mean_return': best_return,
            'std': best_std
        }
        
        print(f"\n{dimension}:")
        print(f"  最优策略: {best_strategy}")
        print(f"  平均年化收益率: {best_return:.2%}")
        print(f"  收益率波动率: {best_std:.2%}")
    
    return best_strategies

best_strategies = find_best_strategies(all_results)

In [ ]:
# 6. 生成JSON格式报告

def generate_json_report(all_results, best_strategies):
    """生成JSON格式的完整报告"""
    print("\n生成JSON格式报告...")
    print("=" * 60)
    
    # 构建报告结构
    report = {
        'task_id': 'T35',
        'task_name': '不同曲线投资顺序分析',
        'generated_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'parameters': {
            'initial_amount': INITIAL_AMOUNT,
            'start_date': '2014-01-01',
            'end_date': '2024-12-08',
            'cycles_count': len(YIELD_DOWN_CYCLES)
        },
        'cycles': [],
        'dimensions': {},
        'best_strategies': best_strategies,
        'output_files': []
    }
    
    # 添加周期信息
    if 'cycles' in all_results:
        for _, row in all_results['cycles'].iterrows():
            report['cycles'].append({
                'cycle': int(row['cycle']),
                'start_date': row['start_date'],
                'end_date': row['end_date'],
                'start_yield': float(row['start_yield']),
                'end_yield': float(row['end_yield']),
                'yield_change': float(row['yield_change'])
            })
    
    # 添加各维度分析结果
    dimensions = ['品种维度', '久期维度', '资质维度']
    
    for dimension in dimensions:
        stats_key = f'{dimension}_stats'
        backtest_key = f'{dimension}_backtest'
        
        if stats_key not in all_results:
            continue
        
        stats_df = all_results[stats_key]
        backtest_df = all_results[backtest_key]
        
        # 策略统计
        strategies = []
        for strategy, row in stats_df.iterrows():
            strategies.append({
                'name': strategy,
                'mean_return': float(row['年化收益率均值']),
                'std': float(row['年化收益率标准差']),
                'min_return': float(row['年化收益率最小值']),
                'max_return': float(row['年化收益率最大值'])
            })
        
        # 按平均收益率排序
        strategies.sort(key=lambda x: x['mean_return'], reverse=True)
        
        report['dimensions'][dimension] = {
            'strategy_count': len(strategies),
            'strategies': strategies
        }
    
    # 添加输出文件列表
    output_files = []
    for f in os.listdir(OUTPUT_DIR):
        file_path = os.path.join(OUTPUT_DIR, f)
        if os.path.isfile(file_path):
            output_files.append({
                'name': f,
                'size': os.path.getsize(file_path),
                'type': f.split('.')[-1]
            })
    
    report['output_files'] = output_files
    
    # 保存JSON报告
    json_file = os.path.join(OUTPUT_DIR, 'T35-分析报告.json')
    with open(json_file, 'w', encoding='utf-8') as f:
        json.dump(report, f, ensure_ascii=False, indent=2)
    
    print(f"JSON报告已保存: {json_file}")
    
    return report

json_report = generate_json_report(all_results, best_strategies)

In [ ]:
# 7. 显示报告摘要

def display_report_summary(report):
    """显示报告摘要"""
    print("\n" + "=" * 60)
    print("T35 - 不同曲线投资顺序分析 - 报告摘要")
    print("=" * 60)
    
    print(f"\n任务ID: {report['task_id']}")
    print(f"任务名称: {report['task_name']}")
    print(f"生成时间: {report['generated_at']}")
    
    print(f"\n参数配置:")
    print(f"  初始投资金额: {report['parameters']['initial_amount']:,.0f} 元")
    print(f"  日期范围: {report['parameters']['start_date']} ~ {report['parameters']['end_date']}")
    print(f"  分析周期数: {report['parameters']['cycles_count']}")
    
    print(f"\n分析周期:")
    for cycle in report['cycles']:
        print(f"  周期{cycle['cycle']}: {cycle['start_date']} ~ {cycle['end_date']} "
              f"(收益率变动 {cycle['yield_change']:.2f}%)")
    
    print(f"\n最优策略:")
    for dimension, info in report['best_strategies'].items():
        print(f"  {dimension}:")
        print(f"    策略: {info['strategy']}")
        print(f"    平均年化收益率: {info['mean_return']:.2%}")
        print(f"    波动率: {info['std']:.2%}")
    
    print(f"\n输出文件统计:")
    file_types = {}
    for f in report['output_files']:
        ext = f['type']
        if ext not in file_types:
            file_types[ext] = {'count': 0, 'size': 0}
        file_types[ext]['count'] += 1
        file_types[ext]['size'] += f['size']
    
    for ext, info in file_types.items():
        print(f"  .{ext}: {info['count']} 个文件, {info['size']:,} bytes")

display_report_summary(json_report)

In [ ]:
# 8. 列出所有输出文件

def list_all_output_files():
    """列出所有输出文件"""
    print("\n所有输出文件")
    print("=" * 60)
    
    all_files = []
    for f in os.listdir(OUTPUT_DIR):
        file_path = os.path.join(OUTPUT_DIR, f)
        if os.path.isfile(file_path):
            all_files.append({
                'name': f,
                'size': os.path.getsize(file_path),
                'modified': datetime.fromtimestamp(os.path.getmtime(file_path)).strftime('%Y-%m-%d %H:%M:%S')
            })
    
    # 按类型分组
    file_groups = {}
    for f in all_files:
        ext = f['name'].split('.')[-1]
        if ext not in file_groups:
            file_groups[ext] = []
        file_groups[ext].append(f)
    
    total_size = 0
    for ext in sorted(file_groups.keys()):
        print(f"\n.{ext} 文件:")
        for f in sorted(file_groups[ext], key=lambda x: x['name']):
            print(f"  - {f['name']} ({f['size']:,} bytes)")
            total_size += f['size']
    
    print(f"\n总计: {len(all_files)} 个文件, {total_size:,} bytes ({total_size/1024/1024:.2f} MB)")

list_all_output_files()

### 总结

本章完成了以下工作：
1. 加载了所有分析结果数据
2. 生成了各维度的详细分析报告
3. 找出了每个维度的最优策略
4. 生成了JSON格式的完整报告

**输出文件:**
- `品种维度策略分析报告.txt`
- `久期维度策略分析报告.txt`
- `资质维度策略分析报告.txt`
- `T35-分析报告.json`

---

## 项目完成

本项目完成了债券投资组合策略分析的全流程：

1. **数据准备**：配置加载、数据库连接测试
2. **周期标注**：生成6个历史收益率下行周期
3. **数据获取**：从数据库获取10种债券类型的收益率数据
4. **策略生成**：生成三个维度共10种策略序列
5. **回测分析**：使用债券收益计算公式进行策略回测
6. **可视化**：生成12张分析图表
7. **报告汇总**：生成详细分析报告

### 核心发现

**债券收益计算公式:**
- 票息收益 = 期初收益率 × (持有天数 / 365)
- 价格收益 = -修正久期 × 收益率变动 × (持有天数 / 365)
- 总收益 = 票息收益 + 价格收益

**三个分析维度:**
- 品种维度：利率债/金融债/信用债的投资顺序
- 久期维度：长久期优先 vs 短久期优先
- 资质维度：高资质优先 vs 弱资质优先